# Report-Language Uncertainty Predicts Radiologist Spatial Disagreement
## End-to-end MedGemma pipeline (Google Colab)

Runs the full pipeline **exclusively with MedGemma** on PadChest-GR:

1. Clone the repo from GitHub.
2. Install dependencies.
3. Log in to Hugging Face (you must have already accepted the MedGemma license at https://huggingface.co/google/medgemma-4b-it).
4. Load + filter PadChest-GR (positive findings with both readers).
5. Classify each sentence with MedGemma (`google/medgemma-4b-it`), batched.
6. Compute reader-reader mask IoU.
7. Run group statistics + Mann-Whitney + bootstrap + permutation tests + per-finding control + OLS regression.
8. Show plots and example tables inline.

**Runtime:** select `Runtime → Change runtime type → A100 GPU` before running.

## 1. Sanity check the GPU

In [ ]:
!nvidia-smi

## 2. Clone the repo

If your repo is private, run `huggingface-cli login` style auth for GitHub first, or use a `git+https://<TOKEN>@github.com/...` URL.

In [ ]:
REPO_URL = 'https://github.com/nprakash1/uncertaintyestimateyang.git'
REPO_DIR = '/content/uncertaintyestimateyang'

import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

PROJECT_DIR = os.path.join(REPO_DIR, 'project')
SRC_DIR = os.path.join(PROJECT_DIR, 'src')
print('Project dir:', PROJECT_DIR)
print('Contents:', os.listdir(PROJECT_DIR))

## 3. Install dependencies

MedGemma is a Gemma-3 model; we need a recent `transformers`.

In [ ]:
!pip install -q -U 'transformers>=4.50' 'accelerate>=0.30' 'sentencepiece' 'huggingface_hub'
!pip install -q -U 'scipy>=1.10' 'scikit-learn>=1.3' 'matplotlib>=3.7' 'pandas>=2.0'

## 4. Hugging Face login

Before running this cell:
1. Create a HF token at https://huggingface.co/settings/tokens.
2. Accept the model license at https://huggingface.co/google/medgemma-4b-it.

You can paste the token interactively *or* set it via Colab Secrets (recommended).

In [ ]:
import os
from huggingface_hub import login

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass('Paste Hugging Face token: ')

login(token=HF_TOKEN)
os.environ['HF_TOKEN'] = HF_TOKEN
print('HF login OK.')

## 5. Run the end-to-end pipeline (MedGemma only)

Batch size 16 fits comfortably on an A100. Increase to 32 or 64 if VRAM allows for additional speedup.

In [ ]:
import os, sys, subprocess, time

os.chdir(SRC_DIR)
print('CWD:', os.getcwd())

BATCH_SIZE = 16  # bump up to 32/64 on A100 80GB

t0 = time.time()
cmd = [sys.executable, 'run_pipeline.py',
       '--scorer', 'medgemma',
       '--batch_size', str(BATCH_SIZE)]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print(f'Total runtime: {(time.time()-t0)/60:.1f} min')

## 6. Inspect the results

All JSON/CSV outputs land in `project/outputs/`; figures in `project/figures/`.

In [ ]:
import json, pandas as pd, pathlib

OUT = pathlib.Path(PROJECT_DIR) / 'outputs'
FIG = pathlib.Path(PROJECT_DIR) / 'figures'
PROC = pathlib.Path(PROJECT_DIR) / 'data' / 'processed'

group_stats = json.loads((OUT / 'group_statistics.json').read_text())
stat_tests  = json.loads((OUT / 'statistical_tests.json').read_text())
regression  = json.loads((OUT / 'regression_results.json').read_text())

print('=== Group statistics ===')
print(json.dumps(group_stats, indent=2))
print('\n=== Statistical tests ===')
print(json.dumps(stat_tests, indent=2))
print('\n=== Regression (control for finding label + box area) ===')
print(json.dumps(regression, indent=2))

In [ ]:
from IPython.display import Image, display, Markdown

display(Markdown('### Reader IoU by uncertainty group'))
display(Image(filename=str(FIG / 'iou_by_uncertainty_group.png')))

display(Markdown('### IoU histogram by group'))
display(Image(filename=str(FIG / 'iou_histogram_by_group.png')))

display(Markdown('### Example grid'))
display(Image(filename=str(FIG / 'example_grid.png')))

In [ ]:
display(Markdown('### Per-finding control analysis'))
pf = pd.read_csv(OUT / 'per_finding_analysis.csv')
display(pf)

display(Markdown('### Top examples: uncertain + low IoU'))
display(pd.read_csv(OUT / 'examples_uncertain_low_iou.csv').head(15))

display(Markdown('### Top examples: certain + high IoU'))
display(pd.read_csv(OUT / 'examples_certain_high_iou.csv').head(15))

## 7. Quick summary of the merged sample table

In [ ]:
merged = pd.read_csv(PROC / 'samples_with_uncertainty_and_iou.csv')
print('rows:', len(merged))
display(merged['uncertainty_label'].value_counts())
display(merged.groupby('uncertainty_label')['reader_iou'].describe())

## 8. (Optional) Download all outputs as a zip

In [ ]:
import shutil
shutil.make_archive('/content/uncertainty_outputs', 'zip',
                    root_dir=PROJECT_DIR,
                    base_dir='.')
print('Saved /content/uncertainty_outputs.zip')
try:
    from google.colab import files
    files.download('/content/uncertainty_outputs.zip')
except Exception:
    pass